# 🏦 Indian Banking Customer Analysis — SQL with DuckDB

**Dataset:** 100,000 Indian bank customers across State Bank of India & Central Bank of India  
**Tech:** DuckDB + Python  
**Columns:** customer_id, bank_name, full_name, gender, date_of_birth, occupation, annual_income, primary_source_of_funds, account_type, account_balance, credit_score, account_status, created_date, last_transaction_date, transactions_per_month, address, email, phone

---

In [ ]:
import duckdb
conn = duckdb.connect('indian_banking.duckdb')

def sql(query):
    return conn.execute(query).fetchdf()

print('Connected! Total customers:', conn.execute('SELECT COUNT(*) FROM customers').fetchone()[0])

## 📊 1. Data Overview & Summary Statistics

In [ ]:
# Bank-wise customer count
sql("SELECT bank_name, COUNT(*) AS customers FROM customers GROUP BY bank_name")

In [ ]:
# Account type distribution
sql("SELECT account_type, COUNT(*) AS count FROM customers GROUP BY account_type ORDER BY count DESC")

In [ ]:
# Active vs Inactive accounts
sql("""
SELECT account_status, COUNT(*) AS count,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentage
FROM customers GROUP BY account_status
""")

In [ ]:
# Overall financial summary
sql("""
SELECT ROUND(AVG(account_balance), 2) AS avg_balance,
       ROUND(AVG(annual_income), 2) AS avg_income,
       ROUND(AVG(credit_score), 2) AS avg_credit_score,
       ROUND(AVG(transactions_per_month), 2) AS avg_txn_per_month
FROM customers
""")

## 👥 2. Demographic Analysis

In [ ]:
# Gender distribution with avg income & balance
sql("""
SELECT gender, COUNT(*) AS customers,
       ROUND(AVG(annual_income), 2) AS avg_income,
       ROUND(AVG(account_balance), 2) AS avg_balance
FROM customers GROUP BY gender
""")

In [ ]:
# Age group analysis
sql("""
SELECT 
    CASE 
        WHEN YEAR(CURRENT_DATE) - YEAR(date_of_birth) < 25 THEN 'Under 25'
        WHEN YEAR(CURRENT_DATE) - YEAR(date_of_birth) BETWEEN 25 AND 35 THEN '25-35'
        WHEN YEAR(CURRENT_DATE) - YEAR(date_of_birth) BETWEEN 36 AND 50 THEN '36-50'
        WHEN YEAR(CURRENT_DATE) - YEAR(date_of_birth) BETWEEN 51 AND 65 THEN '51-65'
        ELSE 'Above 65'
    END AS age_group,
    COUNT(*) AS customers,
    ROUND(AVG(account_balance), 2) AS avg_balance,
    ROUND(AVG(annual_income), 2) AS avg_income
FROM customers
GROUP BY age_group ORDER BY age_group
""")

In [ ]:
# Top 10 occupations by customer count
sql("""
SELECT occupation, COUNT(*) AS customers,
       ROUND(AVG(annual_income), 2) AS avg_income,
       ROUND(AVG(credit_score), 2) AS avg_credit_score
FROM customers GROUP BY occupation
ORDER BY customers DESC LIMIT 10
""")

## 🌍 3. Geographic Analysis

In [ ]:
# Customer distribution by city
sql("""
SELECT TRIM(SPLIT_PART(address, ',', 3)) AS city,
       COUNT(*) AS customers,
       ROUND(AVG(account_balance), 2) AS avg_balance,
       ROUND(SUM(account_balance), 2) AS total_deposits
FROM customers
GROUP BY city ORDER BY customers DESC
""")

## 💰 4. Financial Analysis

In [ ]:
# Income distribution brackets
sql("""
SELECT 
    CASE 
        WHEN annual_income = 0 THEN 'No Income'
        WHEN annual_income < 300000 THEN 'Below 3L'
        WHEN annual_income BETWEEN 300000 AND 1000000 THEN '3-10L'
        WHEN annual_income BETWEEN 1000001 AND 2500000 THEN '10-25L'
        ELSE 'Above 25L'
    END AS income_bracket,
    COUNT(*) AS customers,
    ROUND(AVG(account_balance), 2) AS avg_balance,
    ROUND(AVG(credit_score), 2) AS avg_credit_score
FROM customers GROUP BY income_bracket
ORDER BY avg_balance DESC
""")

In [ ]:
# Income percentiles
sql("""
SELECT 
    ROUND(PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY annual_income), 2) AS p25,
    ROUND(PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY annual_income), 2) AS median,
    ROUND(PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY annual_income), 2) AS p75,
    ROUND(PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY annual_income), 2) AS p90
FROM customers
""")

In [ ]:
# Credit score distribution
sql("""
SELECT 
    CASE 
        WHEN credit_score < 400 THEN 'Poor (<400)'
        WHEN credit_score BETWEEN 400 AND 599 THEN 'Fair (400-599)'
        WHEN credit_score BETWEEN 600 AND 749 THEN 'Good (600-749)'
        ELSE 'Excellent (750+)'
    END AS credit_tier,
    COUNT(*) AS customers,
    ROUND(AVG(annual_income), 2) AS avg_income,
    ROUND(AVG(account_balance), 2) AS avg_balance
FROM customers GROUP BY credit_tier
ORDER BY avg_balance DESC
""")

In [ ]:
# Top 10 highest-paid occupations
sql("""
SELECT occupation,
       ROUND(AVG(annual_income), 2) AS avg_income,
       ROUND(AVG(account_balance), 2) AS avg_balance,
       COUNT(*) AS customers
FROM customers GROUP BY occupation
ORDER BY avg_income DESC LIMIT 10
""")

In [ ]:
# Gender income gap by occupation
sql("""
SELECT occupation,
       ROUND(AVG(CASE WHEN gender='Male' THEN annual_income END), 2) AS male_avg,
       ROUND(AVG(CASE WHEN gender='Female' THEN annual_income END), 2) AS female_avg,
       ROUND(AVG(CASE WHEN gender='Male' THEN annual_income END) -
             AVG(CASE WHEN gender='Female' THEN annual_income END), 2) AS gap
FROM customers GROUP BY occupation
HAVING COUNT(CASE WHEN gender='Male' THEN 1 END) > 50
   AND COUNT(CASE WHEN gender='Female' THEN 1 END) > 50
ORDER BY ABS(gap) DESC LIMIT 10
""")

## 🎯 5. Customer Segmentation

In [ ]:
# RFM-style segmentation (Recency, Frequency, Monetary)
sql("""
SELECT 
    CASE 
        WHEN DATEDIFF('day', last_transaction_date, CURRENT_DATE) <= 30 THEN 'Recent'
        WHEN DATEDIFF('day', last_transaction_date, CURRENT_DATE) <= 90 THEN 'Moderate'
        ELSE 'Dormant'
    END AS recency,
    CASE 
        WHEN transactions_per_month >= 80 THEN 'High Freq'
        WHEN transactions_per_month >= 40 THEN 'Med Freq'
        ELSE 'Low Freq'
    END AS frequency,
    CASE 
        WHEN account_balance >= 1000000 THEN 'High Value'
        WHEN account_balance >= 300000 THEN 'Med Value'
        ELSE 'Low Value'
    END AS monetary,
    COUNT(*) AS customers,
    ROUND(AVG(account_balance), 2) AS avg_balance
FROM customers WHERE account_status = 'Active'
GROUP BY recency, frequency, monetary
ORDER BY customers DESC
""")

In [ ]:
# Wealth distribution by gender
sql("""
SELECT 
    CASE 
        WHEN account_balance < 100000 THEN 'Low (<1L)'
        WHEN account_balance BETWEEN 100000 AND 500000 THEN 'Medium (1-5L)'
        WHEN account_balance BETWEEN 500001 AND 2000000 THEN 'High (5-20L)'
        ELSE 'Very High (>20L)'
    END AS wealth_tier,
    gender, COUNT(*) AS customers,
    ROUND(AVG(annual_income), 2) AS avg_income
FROM customers GROUP BY wealth_tier, gender
ORDER BY wealth_tier, gender
""")

In [ ]:
# High-risk customers: low credit score + high balance
sql("""
SELECT full_name, occupation, credit_score, account_balance,
       CASE 
           WHEN credit_score < 400 THEN 'Very High Risk'
           ELSE 'High Risk'
       END AS risk_level
FROM customers
WHERE credit_score < 500 AND account_balance > 500000
ORDER BY account_balance DESC LIMIT 15
""")

## 📈 6. Trends & Retention

In [ ]:
# Yearly account creation trend with retention
sql("""
SELECT YEAR(created_date) AS year,
       COUNT(*) AS total_accounts,
       SUM(CASE WHEN account_status='Active' THEN 1 ELSE 0 END) AS still_active,
       ROUND(SUM(CASE WHEN account_status='Active' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS retention_pct
FROM customers GROUP BY year ORDER BY year
""")

In [ ]:
# Churn risk scoring
sql("""
SELECT 
    CASE 
        WHEN account_status = 'Inactive' THEN 'Churned'
        WHEN DATEDIFF('day', last_transaction_date, CURRENT_DATE) > 180 THEN 'High Risk'
        WHEN DATEDIFF('day', last_transaction_date, CURRENT_DATE) > 90 THEN 'Medium Risk'
        WHEN transactions_per_month < 10 THEN 'Watch'
        ELSE 'Safe'
    END AS churn_segment,
    COUNT(*) AS customers,
    ROUND(AVG(account_balance), 2) AS avg_balance
FROM customers GROUP BY churn_segment
ORDER BY customers DESC
""")

## 🔍 7. Business Insights

In [ ]:
# Bank performance comparison
sql("""
SELECT bank_name,
       COUNT(*) AS customers,
       ROUND(SUM(account_balance), 2) AS total_deposits,
       ROUND(AVG(account_balance), 2) AS avg_balance,
       ROUND(AVG(credit_score), 2) AS avg_credit_score,
       ROUND(AVG(transactions_per_month), 2) AS avg_txn,
       ROUND(SUM(CASE WHEN account_status='Active' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS active_pct
FROM customers GROUP BY bank_name
""")

In [ ]:
# Premium banking candidates
sql("""
SELECT full_name, occupation, annual_income, account_balance, credit_score
FROM customers
WHERE annual_income > 2000000 AND account_balance > 2000000
  AND credit_score > 700 AND account_status = 'Active'
ORDER BY (annual_income + account_balance) DESC LIMIT 10
""")

In [ ]:
# Inactive high-balance accounts (recovery targets)
sql("""
SELECT full_name, account_balance,
       DATEDIFF('day', last_transaction_date, CURRENT_DATE) AS days_inactive,
       occupation, annual_income
FROM customers
WHERE account_status = 'Inactive' AND account_balance > 500000
ORDER BY account_balance DESC LIMIT 10
""")

In [ ]:
# Most profitable customer segments
sql("""
SELECT occupation,
       COUNT(*) AS customers,
       ROUND(SUM(account_balance), 2) AS total_balance,
       ROUND(AVG(account_balance), 2) AS avg_balance,
       ROUND(AVG(transactions_per_month), 2) AS avg_txn
FROM customers WHERE account_status = 'Active'
GROUP BY occupation HAVING customers > 100
ORDER BY total_balance DESC LIMIT 10
""")

In [ ]:
# Email provider popularity
sql("""
SELECT SPLIT_PART(email, '@', 2) AS provider,
       COUNT(*) AS users,
       ROUND(AVG(annual_income), 2) AS avg_income
FROM customers GROUP BY provider
ORDER BY users DESC
""")

In [ ]:
# Outliers: income Z-score by occupation
sql("""
WITH stats AS (
    SELECT occupation, AVG(annual_income) AS avg_inc, STDDEV(annual_income) AS std_inc
    FROM customers GROUP BY occupation HAVING COUNT(*) > 50
)
SELECT c.full_name, c.occupation, c.annual_income,
       ROUND(s.avg_inc, 2) AS occ_avg,
       ROUND((c.annual_income - s.avg_inc) / NULLIF(s.std_inc, 0), 2) AS z_score
FROM customers c JOIN stats s ON c.occupation = s.occupation
WHERE ABS((c.annual_income - s.avg_inc) / NULLIF(s.std_inc, 0)) > 2
ORDER BY ABS(z_score) DESC LIMIT 15
""")

In [ ]:
conn.close()
print('Done!')